In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import roc_auc_score
from rdkit.ML.Scoring.Scoring import CalcBEDROC
import os

In [30]:
time = 2019
all_disease_df_row = pd.read_excel(f"/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_dw_auc0.8.xlsx", sheet_name="disease")  # Replace with your actual file and sheet name

In [31]:
all_disease_df_row.columns

Index(['method', 'feature', 'auroc', 'rank_ratio', 'bedroc_1', 'bedroc_5',
       'bedroc_10', 'bedroc_30', 'pathway enrich intersection (pred & train)',
       'disease', 'category', 'train_sum', 'test_sum', 'cv para',
       'cv bedroc10', 'cv aucroc', 'valid feature'],
      dtype='object')

In [ ]:
# valid = []
# for d in all_disease_df_row['disease'].unique():
#     subdf = all_disease_df_row[all_disease_df_row['disease']==d]
#     if subdf[subdf['feature'] == 'geo_fused']['auroc'].values.item()>0.6:
#         valid.append(d)
# len(valid)
# all_disease_df = all_disease_df_row[all_disease_df_row['disease'].isin(valid)]
# others = set(all_disease_df_row['disease'].unique())-set(valid)
# all_disease_df_row[all_disease_df_row['disease'].isin(others)]

In [32]:
all_disease_df = all_disease_df_row

In [33]:
all_data = []
sum_test = 0
for d in all_disease_df['disease'].unique():
    subdf = all_disease_df[all_disease_df['disease']==d]

    # Select only relevant metric columns
    weighted_df = subdf[['method', 'disease','feature', 'auroc', 'rank_ratio', 'bedroc_1', 'bedroc_5',
                        'bedroc_10', 'bedroc_30']].copy()

    # Get test sample count for this disease (assuming 'subdf' column stores that)
    test_sum = subdf['test_sum'].unique()
    assert len(test_sum) == 1, "Multiple or no test sample counts found"
    test_sum = test_sum[0]
    sum_test+= test_sum
    # Multiply float-type metric columns by test_sum
    for col in weighted_df.columns:
        if weighted_df[col].dtype == 'float':
            weighted_df[col] = weighted_df[col] * test_sum
    all_data.append(weighted_df)
combined_df = pd.concat(all_data, ignore_index=True)
micro_list = []
for f in combined_df['feature'].unique():
    subdf = combined_df[combined_df['feature']==f]
    float_sums = subdf.select_dtypes(include='float').sum()/sum_test
    float_sums_df = pd.DataFrame([float_sums])
    float_sums_df['feature'] = f
    micro_list.append(float_sums_df)
weighted_avg_df = pd.concat(micro_list, ignore_index=True)

In [34]:
all_disease_df

,method,feature,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,pathway enrich intersection (pred & train),disease,category,train_sum,test_sum,cv para,cv bedroc10,cv aucroc,valid feature
0,random_negative,uniport_ppi_2019,0.890,0.110,0.264,0.493,0.596,0.753,0.176,ICD10_C16,Neoplasms,65,25,"{'C_num': 9, 'gamma': 0.117}",0.723000,0.866000,True
1,random_negative,ppi_2019_dw_40,0.898,0.103,0.208,0.476,0.599,0.756,0.160,ICD10_C16,Neoplasms,65,25,"{'C_num': 1, 'gamma': 0.106}",0.724000,0.873000,True
2,random_negative,uniport_bio,0.812,0.189,0.089,0.244,0.351,0.570,0.288,ICD10_C16,Neoplasms,65,25,"{'C_num': 1, 'gamma': 0.015}",0.550000,0.811000,True
3,random_negative,uniport_esm,0.753,0.247,0.020,0.114,0.212,0.444,0.239,ICD10_C16,Neoplasms,65,25,"{'C_num': 9, 'gamma': 0.085}",0.419000,0.709000,False
4,random_negative,uniport_seq,0.714,0.287,0.011,0.098,0.179,0.394,0.248,ICD10_C16,Neoplasms,65,25,"{'C_num': 27, 'gamma': 0.433}",0.524000,0.784000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,random_negative,uniport_esm,0.620,0.380,0.180,0.407,0.451,0.490,0.138,ICD10_N97,Diseases of the genitourinary system,27,2,"{'C_num': 1, 'gamma': 0.34}",0.719000,0.798000,False
340,random_negative,uniport_seq,0.921,0.079,0.001,0.161,0.347,0.674,0.127,ICD10_N97,Diseases of the genitourinary system,27,2,"{'C_num': 1, 'gamma': 0.866}",0.691000,0.771000,False
341,random_negative,linear_fused,0.965,0.035,0.219,0.487,0.638,0.842,0.124,ICD10_N97,Diseases of the genitourinary system,27,2,{'C': 0.001},0.873000,0.921000,False
342,random_negative,geo_fused,0.959,0.041,0.307,0.494,0.618,0.821,0.119,ICD10_N97,Diseases of the genitourinary system,27,2,{'C': 0.001},0.800000,0.897000,False


In [35]:
weighted_cat = []
for c in all_disease_df['category'].unique():
    all_data = []
    sum_test = 0
    disease_list = all_disease_df[all_disease_df['category']==c]['disease'].unique()
    for d in disease_list:
        subdf = all_disease_df[all_disease_df['disease']==d]

        # Select only relevant metric columns
        weighted_df = subdf[['method', 'disease','feature', 'auroc', 'rank_ratio', 'bedroc_1', 'bedroc_5',
                            'bedroc_10', 'bedroc_30']].copy()

        # Get test sample count for this disease (assuming 'subdf' column stores that)
        test_sum = subdf['test_sum'].unique()
        assert len(test_sum) == 1, "Multiple or no test sample counts found"
        test_sum = test_sum[0]
        sum_test+= test_sum
        # Multiply float-type metric columns by test_sum
        for col in weighted_df.columns:
            if weighted_df[col].dtype == 'float':
                weighted_df[col] = weighted_df[col] * test_sum
        all_data.append(weighted_df)
    combined_df = pd.concat(all_data, ignore_index=True)
    micro_list = []
    for f in combined_df['feature'].unique():
        subdf = combined_df[combined_df['feature']==f]
        float_sums = subdf.select_dtypes(include='float').sum()/sum_test
        float_sums_df = pd.DataFrame([float_sums])
        float_sums_df['feature'] = f
        micro_list.append(float_sums_df)
    micro_df = pd.concat(micro_list, ignore_index=True)
    micro_df['category'] = c
    weighted_cat.append(micro_df)
weighted_cat_df = pd.concat(weighted_cat, ignore_index=True)

In [36]:
weighted_cat_df

,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,feature,category
0,0.841790,0.158210,0.262935,0.453516,0.548500,0.699306,uniport_ppi_2019,Neoplasms
1,0.861145,0.139258,0.238565,0.469242,0.576742,0.719371,ppi_2019_dw_40,Neoplasms
2,0.746903,0.253952,0.094403,0.235258,0.327177,0.504516,uniport_bio,Neoplasms
3,0.741016,0.259435,0.029161,0.127790,0.216855,0.437613,uniport_esm,Neoplasms
4,0.732177,0.268226,0.037210,0.151500,0.235887,0.436032,uniport_seq,Neoplasms
...,...,...,...,...,...,...,...,...
83,0.576000,0.424000,0.064400,0.197750,0.265150,0.366900,uniport_esm,Diseases of the genitourinary system
84,0.611650,0.388100,0.050050,0.157400,0.229700,0.365700,uniport_seq,Diseases of the genitourinary system
85,0.733250,0.266900,0.104200,0.184650,0.258450,0.442350,linear_fused,Diseases of the genitourinary system
86,0.754500,0.245500,0.095150,0.207350,0.286800,0.463600,geo_fused,Diseases of the genitourinary system


In [37]:
combined_df = weighted_cat_df[['feature', 'auroc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30','category']]
# Define alternating background colors per disease
base_colors = ['#ffdddd', '#dbf7db']


# Build mapping for background colors per disease
category = combined_df['category'].unique()
color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(category)}

target_cols = ['auroc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30']

def combined_style2(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)

    # Add background color row-wise
    for idx, row in df.iterrows():
        bg_color = color_map[row['category']]
        styles.loc[idx, :] = f'background-color: {bg_color};'

    # Highlight max in each group & each target column
    for category, group in df.groupby('category'):
        for col in target_cols:
            max_val = group[col].max()
            max_indices = group[group[col] == max_val].index
            for idx in max_indices:
                styles.loc[idx, col] += ' color: red; font-weight: bold;'
    
    return styles


numeric_cols = combined_df.select_dtypes(include=['number']).columns

# Apply styling and format ONLY numeric columns
styled_df = (
    combined_df.style
    .apply(combined_style2, axis=None)
    .format({col: "{:.3f}" for col in numeric_cols})
)

styled_df

,feature,auroc,bedroc_1,bedroc_5,bedroc_10,bedroc_30,category
0,uniport_ppi_2019,0.842,0.263,0.454,0.548,0.699,Neoplasms
1,ppi_2019_dw_40,0.861,0.239,0.469,0.577,0.719,Neoplasms
2,uniport_bio,0.747,0.094,0.235,0.327,0.505,Neoplasms
3,uniport_esm,0.741,0.029,0.128,0.217,0.438,Neoplasms
4,uniport_seq,0.732,0.037,0.152,0.236,0.436,Neoplasms
5,linear_fused,0.849,0.240,0.452,0.556,0.703,Neoplasms
6,geo_fused,0.861,0.200,0.415,0.539,0.706,Neoplasms
7,early_fusion,0.741,0.029,0.128,0.217,0.438,Neoplasms
8,uniport_ppi_2019,0.915,0.234,0.414,0.512,0.708,Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism
9,ppi_2019_dw_40,0.913,0.186,0.435,0.544,0.721,Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism


In [38]:
with pd.ExcelWriter(f'/itf-fi-ml/shared/users/ziyuzh/svm/results/{str(time)}_dw_auc0.8.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    styled_df.to_excel(writer, sheet_name='category (weighted avg)', index=False)
    weighted_avg_df.to_excel(writer, sheet_name='all (weighted avg)', index=False)

In [2]:
all_results = dict()
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2017_cv_rank_save_pred/'
for file in os.listdir(root):
    filepath = os.path.join(root,file)
    disease_id = file[:9]
    with open(filepath, 'rb') as file:
        disease_pred = pickle.load(file)
    all_results[disease_id] = disease_pred

In [3]:
feature_names = []
for i in list(next(iter(all_results.values())).keys()):
    feature_names.append(i)

In [4]:
results_dict = {key: [] for key in feature_names}
for disease, preds in all_results.items():
    if len(preds) == len(feature_names):
        for feature in feature_names:
            results_dict[feature].append(preds[feature])
    else:
        print(disease)

ICD10_F90


In [19]:
micro_auc_dict = dict()
bedroc_dict = dict()
y_true_concat = np.concatenate(results_dict['true_label'])
for feature in feature_names[1:]:
    y_pred_concat = np.concatenate(results_dict[feature])
    auc = roc_auc_score(y_true_concat, y_pred_concat)
    micro_auc_dict[feature] = auc
    scores = np.column_stack((y_true_concat, y_pred_concat))  # Stack labels and scores as columns
    scores = scores[scores[:, 1].argsort()[::-1]]
    bedroc_dict[feature] = [CalcBEDROC(scores, col=0, alpha=160.9), 
                            CalcBEDROC(scores, col=0, alpha=32.2),
                            CalcBEDROC(scores, col=0, alpha=16.1),
                            CalcBEDROC(scores, col=0, alpha=5.3)]
bedroc = pd.DataFrame.from_dict(bedroc_dict, orient='index')
# Optional: name the columns
bedroc.columns = ['bedroc_1', 'bedroc_5', 'bedroc_10','bedroc_30']
bedroc['micro_auc'] = list(micro_auc_dict.values())
ordera = ['ppi_2017_dw_80', 'uniport_ppi_2017', 'uniport_exp', 'uniport_seq',
       'uniport_esm', 'early_fusion', 'linear_fused', 'geo_fused',
       'weighted_linear_fused', 'weighted_geo_fused']

bedroc = bedroc.reset_index().rename(columns={'index': 'feature'})

# Step 2: Reorder rows based on feature_list
bedroc = bedroc.set_index('feature').loc[ordera].reset_index()
bedroc = bedroc.round(3)
desired_order = ['feature', 'micro_auc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30']
bedroc = bedroc[desired_order]

In [13]:
icd_dict = {
    'Certain infectious and parasitic diseases': ['A00','B99'],
    'Neoplasms': ['C00','D48'],
    'Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism': ['D50','D89'],
    'Endocrine, nutritional and metabolic diseases': ['E00','E90'],
    'Mental and behavioural disorders': ['F00','F99'],
    'Diseases of the nervous system': ['G00','G99'],
    'Diseases of the eye and adnexa': ['H00','H59'],
    'Diseases of the ear and mastoid process': ['H60','H95'],
    'Diseases of the circulatory system': ['I00','I99'],
    'Diseases of the respiratory system': ['J00','J99'],
    'Diseases of the digestive system': ['K00','K93'],
    'Diseases of the skin and subcutaneous tissue': ['L00','L99'],
    'Diseases of the musculoskeletal system and connective tissue': ['M00','M99'],
    'Diseases of the genitourinary system': ['N00','N99']
}

def find_disease_category(icd_code):
    icd_num = icd_code.split('_')[1]  # Extract ICD-10 code
    # print(icd_num)
    icd_letter = icd_num[0]  # Extract first letter (C, D, etc.)
    icd_number = int(icd_num[1:])  # Extract numeric part

    for category, (start, end) in icd_dict.items():
        start_letter, start_num = start[0], int(start[1:])
        end_letter, end_num = end[0], int(end[1:])

        if start_letter <= icd_letter <= end_letter:  # Ensure it's within the letter range
            if start_letter == icd_letter and start_num <= icd_number:
                return category
            if end_letter == icd_letter and icd_number <= end_num:
                return category
            if start_letter < icd_letter < end_letter:
                return category  # Covers ranges like C00-D48
        
    return 'Unknown Category'
category_map = dict()
for disease in all_results.keys():
    if 'ICD' in disease:
        category_map[disease] = find_disease_category(disease)

In [14]:
from collections import defaultdict

# Create a defaultdict of lists
inverted = defaultdict(list)

# Group keys by their values
for key, value in category_map.items():
    inverted[value].append(key)

# Optional: convert to regular dict
inverted_dict = dict(inverted)

In [15]:
category_results = []
for category in inverted_dict.keys():
    sub_results = dict()
    for k in inverted_dict[category]:
        if k in all_results.keys():
            sub_results[k] = all_results[k]

    results_dict = {key: [] for key in feature_names}
    for disease, preds in sub_results.items():
        if len(preds) == len(feature_names):
            for feature in feature_names:
                results_dict[feature].append(preds[feature])
        else:
            print(disease)

    micro_auc_dict = dict()
    bedroc_dict = dict()
    y_true_concat = np.concatenate(results_dict['true_label'])
    for feature in feature_names[1:]:
        y_pred_concat = np.concatenate(results_dict[feature])
        auc = roc_auc_score(y_true_concat, y_pred_concat)
        micro_auc_dict[feature] = auc
        scores = np.column_stack((y_true_concat, y_pred_concat))  # Stack labels and scores as columns
        scores = scores[scores[:, 1].argsort()[::-1]]
        bedroc_dict[feature] = [CalcBEDROC(scores, col=0, alpha=160.9), 
                                CalcBEDROC(scores, col=0, alpha=32.2),
                                CalcBEDROC(scores, col=0, alpha=16.1),
                                CalcBEDROC(scores, col=0, alpha=5.3)]
    bedroc = pd.DataFrame.from_dict(bedroc_dict, orient='index')
    # Optional: name the columns
    bedroc.columns = ['bedroc_1', 'bedroc_5', 'bedroc_10','bedroc_30']
    bedroc['micro_auc'] = list(micro_auc_dict.values())
    bedroc = bedroc.round(3)
    bedroc['category'] = category
    category_results.append(bedroc)

ICD10_F90


In [16]:
combined_df = pd.concat(category_results, ignore_index=False)

In [17]:
combined_df = combined_df.reset_index().rename(columns={'index': 'feature'})

In [22]:
combined_df.columns

Index(['feature', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30',
       'micro_auc', 'category'],
      dtype='object')

In [ ]:
combined_df = combined_df[['feature', 'micro_auc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30','category']]
# Define alternating background colors per disease
base_colors = ['#ffdddd', '#dbf7db']


# Build mapping for background colors per disease
category = combined_df['category'].unique()
color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(category)}

target_cols = ['micro_auc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30']

def combined_style2(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)

    # Add background color row-wise
    for idx, row in df.iterrows():
        bg_color = color_map[row['category']]
        styles.loc[idx, :] = f'background-color: {bg_color};'

    # Highlight max in each group & each target column
    for category, group in df.groupby('category'):
        for col in target_cols:
            max_val = group[col].max()
            max_indices = group[group[col] == max_val].index
            for idx in max_indices:
                styles.loc[idx, col] += ' color: red; font-weight: bold;'
    
    return styles


numeric_cols = combined_df.select_dtypes(include=['number']).columns

# Apply styling and format ONLY numeric columns
styled_df = (
    combined_df.style
    .apply(combined_style2, axis=None)
    .format({col: "{:.3f}" for col in numeric_cols})
)

styled_df

,feature,micro_auc,bedroc_1,bedroc_5,bedroc_10,bedroc_30,category
0,uniport_ppi_2017,0.714,0.100,0.216,0.276,0.425,Neoplasms
1,ppi_2017_dw_80,0.709,0.137,0.235,0.293,0.437,Neoplasms
2,uniport_exp,0.581,0.045,0.103,0.161,0.314,Neoplasms
3,uniport_seq,0.676,0.045,0.145,0.219,0.395,Neoplasms
4,uniport_esm,0.687,0.043,0.141,0.220,0.402,Neoplasms
5,early_fusion,0.679,0.043,0.131,0.209,0.392,Neoplasms
6,linear_fused,0.680,0.161,0.254,0.328,0.487,Neoplasms
7,geo_fused,0.684,0.147,0.244,0.313,0.476,Neoplasms
8,weighted_linear_fused,0.681,0.160,0.254,0.327,0.486,Neoplasms
9,weighted_geo_fused,0.683,0.144,0.247,0.315,0.476,Neoplasms
